# 05 - Modelo Final

Treino final escolhido pelo conjunto de validacao. O conjunto de teste deve ser avaliado uma unica vez e os resultados finais sao salvos em arquivos auditaveis.

In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

def ensure_package(package, import_name=None):
    import_name = import_name or package.split('==')[0].replace('-', '_')
    if importlib.util.find_spec(import_name) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

if IN_COLAB:
    ensure_package('medmnist==3.0.2', 'medmnist')
    ensure_package('pandas', 'pandas')
    ensure_package('matplotlib', 'matplotlib')
    ensure_package('seaborn', 'seaborn')

project_candidates = [Path.cwd(), Path.cwd().parent, Path('/content/AP2_IA'), Path('/content/ap2-ia'), Path('/content/drive/MyDrive/AP2_IA'), Path('/kaggle/working/ap2-ia'), Path('/root/ap2-ia')]
PROJECT_ROOT = next((path for path in project_candidates if (path / 'src' / 'train.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Nao encontrei a raiz do projeto. Execute o notebook dentro da pasta do repositorio.')
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from torch import nn
from torch.optim import AdamW, SGD

from data.dataset import PATHMNIST_CLASSES, get_loaders
from evaluate import predict_loader, save_classification_artifacts
from models.custom_cnn import CustomCNN
from models.transfer import create_model, parameter_groups
from train import run_epoch
from utils import EarlyStopping, Timer, collect_hardware_info, device, save_json, set_seed

SEED = 42
set_seed(SEED)
DEVICE = device()
FINAL_DIR = PROJECT_ROOT / 'experiments' / 'final'
CHECKPOINT_DIR = PROJECT_ROOT / 'checkpoints' / 'final'
FIGURE_DIR = PROJECT_ROOT / 'outputs' / 'figures'
for path in [FINAL_DIR, CHECKPOINT_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)
save_json(PROJECT_ROOT / 'experiments' / 'hardware.json', collect_hardware_info())
print('Projeto:', PROJECT_ROOT)
print('Dispositivo:', DEVICE)

In [ ]:
results_path = PROJECT_ROOT / 'experiments' / 'stage03_best_by_run.csv'
if results_path.exists() and not pd.read_csv(results_path).empty:
    best = pd.read_csv(results_path).sort_values('acc_val', ascending=False).iloc[0]
    MODEL_NAME = str(best['modelo'])
    MODE = str(best['modo'])
    OPTIMIZER_NAME = str(best['otimizador'])
    LR = float(best['lr'])
else:
    MODEL_NAME = 'resnet50'
    MODE = 'fine_tuning'
    OPTIMIZER_NAME = 'adamw'
    LR = 1e-4

FAST_FINAL_TRAINING = True
EPOCHS = 5 if FAST_FINAL_TRAINING else 30
BATCH_SIZE = 16 if FAST_FINAL_TRAINING else 32
IMAGE_SIZE = 224
NUM_WORKERS = 2 if torch.cuda.is_available() else 0
LABEL_SMOOTHING = 0.1
PATIENCE = 5
AUGMENT_POLICY = 'randaugment'
CHECKPOINT_PATH = CHECKPOINT_DIR / 'best_final_model.pt'

config = {
    'model': MODEL_NAME,
    'mode': MODE,
    'optimizer': OPTIMIZER_NAME,
    'lr': LR,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': IMAGE_SIZE,
    'label_smoothing': LABEL_SMOOTHING,
    'patience': PATIENCE,
    'augment_policy': AUGMENT_POLICY,
    'seed': SEED,
}
save_json(FINAL_DIR / 'final_training_config.json', config)
config

In [ ]:
if MODEL_NAME == 'custom_cnn':
    model = CustomCNN(num_classes=9).to(DEVICE)
    optimizer_params = model.parameters()
else:
    model = create_model(MODEL_NAME, num_classes=9, pretrained=True, mode=MODE).to(DEVICE)
    optimizer_params = parameter_groups(model, LR, MODE)

loaders = get_loaders(batch_size=BATCH_SIZE, image_size=IMAGE_SIZE, num_workers=NUM_WORKERS, augment_policy=AUGMENT_POLICY)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
if OPTIMIZER_NAME == 'sgd':
    optimizer = SGD(optimizer_params, lr=LR, momentum=0.9, nesterov=True, weight_decay=1e-4)
else:
    optimizer = AdamW(optimizer_params, lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
stopper = EarlyStopping(patience=PATIENCE, mode='min')
print('Modelo final:', MODEL_NAME, '| modo:', MODE, '| augment:', AUGMENT_POLICY)

In [ ]:
history = []
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    with Timer() as timer:
        train_loss, train_acc = run_epoch(model, loaders['train'], criterion, optimizer, DEVICE)
        val_loss, val_acc = run_epoch(model, loaders['val'], criterion, None, DEVICE)
        scheduler.step()

    row = {
        'epoch': epoch,
        'loss_train': train_loss,
        'loss_val': val_loss,
        'acc_train': train_acc,
        'acc_val': val_acc,
        'lr': scheduler.get_last_lr()[0],
        'tempo_s': timer.elapsed,
        'vram_mb': torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0.0,
    }
    history.append(row)
    print(f"Epoca {epoch:02d} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'model_state_dict': model.state_dict(),
            'model': MODEL_NAME,
            'mode': MODE,
            'num_classes': 9,
            'epoch': epoch,
            'val_loss': val_loss,
            'val_acc': val_acc,
            'config': config,
        }, CHECKPOINT_PATH)

    if stopper.step(val_loss):
        print('Early stopping acionado.')
        break

history_df = pd.DataFrame(history)
history_df.to_csv(FINAL_DIR / 'final_training_history.csv', index=False)
history_df.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
history_df.plot(x='epoch', y=['loss_train', 'loss_val'], ax=axes[0], title='Modelo final - loss')
history_df.plot(x='epoch', y=['acc_train', 'acc_val'], ax=axes[1], title='Modelo final - accuracy')
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURE_DIR / 'stage05_final_training_curves.png', dpi=160)
fig

In [ ]:
ALLOW_REEVALUATE_TEST = False
metrics_path = FINAL_DIR / 'test_metrics.json'
if metrics_path.exists() and not ALLOW_REEVALUATE_TEST:
    raise RuntimeError('Metricas de teste ja existem. Para evitar reuso do test set, mantenha este resultado ou mude ALLOW_REEVALUATE_TEST=True apenas com justificativa no relatorio.')

checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
y_true, y_pred = predict_loader(model, loaders['test'], DEVICE)
metrics = save_classification_artifacts(y_true, y_pred, PATHMNIST_CLASSES, FINAL_DIR, prefix='test')
save_json(FINAL_DIR / 'test_evaluation_protocol.json', {'test_used_once': True, 'checkpoint': str(CHECKPOINT_PATH), 'best_epoch': int(checkpoint['epoch'])})
metrics

In [ ]:
cm = pd.read_csv(FINAL_DIR / 'test_confusion_matrix.csv', index_col=0)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicao')
plt.ylabel('Classe real')
plt.title('Matriz de confusao 9x9 - teste')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'stage05_test_confusion_matrix.png', dpi=160)

Arquivos gerados: `experiments/final/final_training_history.csv`, `checkpoints/final/best_final_model.pt`, `experiments/final/test_metrics.json`, `experiments/final/test_classification_report.csv`, `experiments/final/test_confusion_matrix.csv` e `outputs/figures/stage05_test_confusion_matrix.png`.